# Setup Data in Local Repository

Large files live in `data/local/` (ignored by git). Small derived files live in `data/tracked/` (committed).

```
data/
├── local/                    # ignored by git
│   ├── fma_small.zip
│   ├── fma_metadata.zip
│   ├── fma_metadata/
│   ├── fma_small_mp3/
│   └── spectrograms/         # later
└── tracked/                  # committed to git
    └── track_metadata.csv
```

## Steps before running this notebook

1. Download `fma_small.zip` and `fma_metadata.zip` from the [FMA repository](https://github.com/mdeff/fma).
2. Create the folder `data/local/` and put both zip files there:
   - `data/local/fma_small.zip`
   - `data/local/fma_metadata.zip`
3. Unzip `fma_metadata.zip` into `data/local/fma_metadata/` (it should contain `tracks.csv`).
4. Confirm `.gitignore` contains `data/local/`.
5. Run the cells below. They extract the songs and write `data/tracked/track_metadata.csv`.

**Note:** `fma_small.zip` is ~7 GiB. Make sure you have ~15 GiB free (zip + extracted files).

Importh and set up the file paths

In [2]:
import zipfile
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data")
LOCAL_DIR = DATA_DIR / "local"        # big files, not in git
TRACKED_DIR = DATA_DIR / "tracked"    # small files, in git

ZIP_PATH = LOCAL_DIR / "fma_small.zip"
EXTRACT_DIR = LOCAL_DIR / "fma_small_mp3"
METADATA_PATH = LOCAL_DIR / "fma_metadata" / "tracks.csv"

# Fail early with a clear message if something's missing
for p in (ZIP_PATH, METADATA_PATH):
    assert p.exists(), f"Missing {p}. See the setup steps above."

Count Songs in ZIP

In [3]:
with zipfile.ZipFile(ZIP_PATH) as z:
    names = [n for n in z.namelist() if n.endswith(".mp3")]

print(len(names), "songs")   # expect 8000
print(names[:3])

8000 songs
['fma_small/000/000002.mp3', 'fma_small/000/000005.mp3', 'fma_small/000/000010.mp3']


Build a table with track ID as the key

In [4]:
songs = pd.DataFrame({"zip_path": names})
songs["track_id"] = songs["zip_path"].apply(lambda p: int(Path(p).stem))
songs = songs.set_index("track_id").sort_index()
songs.head()

,zip_path
track_id,
2,fma_small/000/000002.mp3
5,fma_small/000/000005.mp3
10,fma_small/000/000010.mp3
140,fma_small/000/000140.mp3
141,fma_small/000/000141.mp3


Extract each song

In [5]:
if not EXTRACT_DIR.exists():
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(EXTRACT_DIR)

songs["path"] = songs["zip_path"].apply(lambda p: EXTRACT_DIR / p)

# Confirm the extracted files are where we expect
missing = [p for p in songs["path"] if not p.exists()]
print(f"{len(missing)} missing files")   # expect 0

0 missing files


Load the metadata

In [6]:
tracks = pd.read_csv(
    METADATA_PATH,
    index_col=0,
    header=[0, 1],   # tracks.csv has a two-row header
)

small = tracks[tracks["set", "subset"] == "small"]
print(small.shape)   # expect 8000 rows

small[[("track", "title"), ("artist", "name"), ("track", "genre_top")]].head()

(8000, 52)


,track,artist,track
,title,name,genre_top
track_id,,,
2,Food,AWOL,Hip-Hop
5,This World,AWOL,Hip-Hop
10,Freeway,Kurt Vile,Pop
140,Queen Of The Wires,Alec K. Redfearn & the Eyesores,Folk
141,Ohio,Alec K. Redfearn & the Eyesores,Folk


Join with the tracks table

In [6]:
cols = {
    ("track", "title"): "title",
    ("artist", "name"): "artist",
    ("track", "genre_top"): "genre",
}

small_meta = small[list(cols.keys())].copy()
small_meta.columns = list(cols.values())

meta = songs.join(small_meta)
meta.head()

,zip_path,path,title,artist,genre
track_id,,,,,
2,fma_small/000/000002.mp3,../data/local/fma_small_mp3/fma_small/000/0000...,Food,AWOL,Hip-Hop
5,fma_small/000/000005.mp3,../data/local/fma_small_mp3/fma_small/000/0000...,This World,AWOL,Hip-Hop
10,fma_small/000/000010.mp3,../data/local/fma_small_mp3/fma_small/000/0000...,Freeway,Kurt Vile,Pop
140,fma_small/000/000140.mp3,../data/local/fma_small_mp3/fma_small/000/0001...,Queen Of The Wires,Alec K. Redfearn & the Eyesores,Folk
141,fma_small/000/000141.mp3,../data/local/fma_small_mp3/fma_small/000/0001...,Ohio,Alec K. Redfearn & the Eyesores,Folk


Save the summary table to tracked data folder

In [7]:
TRACKED_DIR.mkdir(parents=True, exist_ok=True)

meta_out = meta[["zip_path", "title", "artist", "genre"]]
meta_out.to_csv(TRACKED_DIR / "track_metadata.csv")

print("Saved", TRACKED_DIR / "track_metadata.csv")

Saved ../data/tracked/track_metadata.csv
